# Notebook 20 — Connections and Ownership Identity

## Bounded question

> What do the runner-level `jockey`, `trainer` and `owner` fields represent in the source, how stable and complete are their labels, and which source-internal identity relationships can be preserved safely without inventing equivalence between people, partnerships, syndicates or organisations?

## Initial governed scope

This notebook investigates three runner-level source-text fields:

- `jockey`
- `trainer`
- `owner`

The source-field governance register assigns all three to the `connections_and_ownership` family, requires their raw values to be preserved, treats blanks as `field_not_supplied`, and leaves their semantics pending.

The investigation begins with source lineage and the governed population only. At this stage, no assumption is made that:

- identical strings always identify the same person or organisation;
- different strings always identify different entities;
- initials uniquely identify a person;
- punctuation changes imply equivalence;
- titles or suffixes can be removed globally;
- jockey and trainer names follow identical identity rules;
- owner partnership or syndicate labels can be decomposed safely;
- a person, organisation, ownership account, partnership or syndicate belongs in one undifferentiated entity model;
- chronology or string similarity proves identity.

Raw source labels, parsed display labels, exact-label identity, provisional source-internal identities, externally verified entities and role assertions will remain separate concepts. Any later normalization must be reversible and preserve source lineage, role context, confidence and review status.

## Stage 1 — Source lineage and governed population

This stage establishes the immutable source, read-only controls, complete governed runner population and provisional race key before interpreting any connection or ownership label.

The source is:

- database: `data/raw/form_2015-present/form_2015-present/raceform.db`
- table: `data`
- governed row predicate: `rowid <> 1`
- provisional race identity: `date + course + off`

The established source population is expected to contain:

- 1,851,285 governed runner rows;
- 189,043 provisional races;
- 37 source columns.

The first code cell opens SQLite in read-only mode, confirms the source schema, reconciles the governed runner and provisional-race counts, and confirms that `jockey`, `trainer` and `owner` are present. It does not trim, parse, normalize, classify or match any name.


In [ ]:
from pathlib import Path
import sqlite3

import pandas as pd


# Resolve the immutable source explicitly from the notebook directory.
PROJECT_ROOT = Path.cwd().resolve().parent
SOURCE_DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"
RACE_KEY_COLUMNS = ["date", "course", "off"]
CONNECTION_IDENTITY_FIELDS = ["jockey", "trainer", "owner"]

EXPECTED_RUNNER_ROWS = 1_851_285
EXPECTED_PROVISIONAL_RACES = 189_043
EXPECTED_SOURCE_COLUMNS = 37

if not SOURCE_DB_PATH.exists():
    raise FileNotFoundError(f"Source database not found: {SOURCE_DB_PATH}")

# Open SQLite read-only so the notebook cannot mutate the source database.
connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    schema = pd.read_sql_query(f"PRAGMA table_info({SOURCE_TABLE})", connection)
    source_columns = schema["name"].tolist()

    missing_fields = [
        field
        for field in CONNECTION_IDENTITY_FIELDS
        if field not in source_columns
    ]
    if missing_fields:
        raise AssertionError(
            f"Missing connections-and-ownership fields: {missing_fields}"
        )

    runner_rows = connection.execute(
        f"SELECT COUNT(*) FROM {SOURCE_TABLE} WHERE {DATA_ROW_PREDICATE}"
    ).fetchone()[0]

    provisional_races = connection.execute(
        f"""
        SELECT COUNT(*)
        FROM (
            SELECT DISTINCT date, course, off
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
        )
        """
    ).fetchone()[0]
finally:
    connection.close()

assert runner_rows == EXPECTED_RUNNER_ROWS
assert provisional_races == EXPECTED_PROVISIONAL_RACES
assert len(source_columns) == EXPECTED_SOURCE_COLUMNS

source_lineage_summary = pd.DataFrame(
    [
        ("source database", SOURCE_DB_PATH.relative_to(PROJECT_ROOT).as_posix()),
        ("source table", SOURCE_TABLE),
        ("data-row predicate", DATA_ROW_PREDICATE),
        ("runner rows", runner_rows),
        ("provisional races", provisional_races),
        ("source columns", len(source_columns)),
        ("provisional race key", " + ".join(RACE_KEY_COLUMNS)),
        (
            "connections-and-ownership fields present",
            ", ".join(CONNECTION_IDENTITY_FIELDS),
        ),
    ],
    columns=["measure", "value"],
)

print("Governed source population confirmed")
source_lineage_summary


## Stage 2 — Confirm inherited source-field governance

Before profiling the contents of the three fields, this stage reads their existing rows from `data/reference/source_field_governance.csv`.

The check is limited to confirming the inherited starting position:

- each field is recorded at runner grain;
- each belongs to `connections_and_ownership`;
- each is declared as source `TEXT`;
- raw preservation is required;
- blanks retain the governed meaning `field_not_supplied`;
- semantic status remains pending;
- the existing Notebook 10 attribution is preserved.

This stage does not revise the register or infer identity from the field labels themselves.


In [ ]:
SOURCE_FIELD_GOVERNANCE_PATH = (
    PROJECT_ROOT / "data" / "reference" / "source_field_governance.csv"
)

if not SOURCE_FIELD_GOVERNANCE_PATH.exists():
    raise FileNotFoundError(
        "Source-field governance register not found: "
        f"{SOURCE_FIELD_GOVERNANCE_PATH}"
    )

source_field_governance = pd.read_csv(SOURCE_FIELD_GOVERNANCE_PATH)

connections_governance = (
    source_field_governance.loc[
        source_field_governance["source_field"].isin(CONNECTION_IDENTITY_FIELDS),
        [
            "ordinal",
            "source_field",
            "declared_type",
            "grain",
            "field_family",
            "raw_preservation",
            "blank_policy",
            "dash_policy",
            "zero_policy",
            "governed_by",
            "status",
        ],
    ]
    .sort_values("ordinal")
    .reset_index(drop=True)
)

assert connections_governance["source_field"].tolist() == CONNECTION_IDENTITY_FIELDS
assert connections_governance["declared_type"].eq("TEXT").all()
assert connections_governance["grain"].eq("runner").all()
assert connections_governance["field_family"].eq(
    "connections_and_ownership"
).all()
assert connections_governance["raw_preservation"].eq("required").all()
assert connections_governance["blank_policy"].eq("field_not_supplied").all()
assert connections_governance["dash_policy"].eq("not_expected").all()
assert connections_governance["zero_policy"].eq("contextual_value").all()
assert connections_governance["governed_by"].eq("Notebook 10").all()
assert connections_governance["status"].eq("pending_semantics").all()

print("Inherited source-field governance confirmed")
connections_governance


## Stage 3 — Next action

Stop after inspecting the outputs from Stages 1 and 2.

The next stage will profile the raw physical behaviour of `jockey`, `trainer` and `owner` independently. It will measure completeness, exact-label vocabulary, temporal coverage, repetition and bounded string features without cleaning or entity matching.

Do not add the Stage 3 query until the Stage 1 and Stage 2 outputs have been returned and reconciled.
